## Projekt 2
**Cel:** Automatyczna analiza obrazu mikroskopowego z wykorzystaniem algorytmu wododziałowego (Watershed) do rozdzielania stykających się obiektów.

### Wprowadzenie
Twoim zadaniem jest napisanie skryptu w roli inżyniera oprogramowania medycznego. Zadania skryptu:
1. Wczytać obraz komórek pod mikroskopem.
2. Oddzielić komórki od tła (progowanie).
3. Rozdzielić stykające się ze sobą komórki za pomocą algorytmu **Watershed**.
4. Wyliczyć z gotowej maski cechy fizyczne (np. liczbę komórek i ich średnią wielkość).

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters, morphology, measure, segmentation, color
from scipy import ndimage as ndi

# Konfiguracja wielkości wykresów
plt.rcParams['figure.figsize'] = (8, 8)

### Krok 1: Wczytanie danych
Wykorzystamy wbudowany zbiór z biblioteki `skimage` (`human_mitosis`), który świetnie nadaje się do zadania separacji nakładających się komórek.

**Zadanie:** Wczytaj obraz, wypisz jego wymiary (shape) i wyświetl go.

In [8]:
# Wczytanie obrazu
image = data.human_mitosis()

# WYKONAJ: Wyświetl oryginalny obraz w skali szarości
# === TWÓJ KÓD TUTAJ ===

# ======================

### Krok 2: Wstępna segmentacja (Otsu)
**Zadanie:** Użyj filtru Gaussa (aby zredukować mikroszum tła), a następnie znajdź optymalny próg metodą Otsu i stwórz binarną maskę (1 - komórka, 0 - tło).

In [ ]:
# WYKONAJ: Rozmycie Gaussa, progowanie Otsu i maska binarna (binary_mask)
# === TWÓJ KÓD TUTAJ ===
binary_mask = np.zeros_like(image) # <- Podmień tę zmienną!
# ======================

# plt.title('Maska po progowaniu Otsu')
# plt.imshow(binary_mask, cmap='gray')
# plt.show()

### Krok 3: Oczyszczenie morfologiczne
Na masce prawdopodobnie zostały drobne białe punkty (fałszywe detekcje na tle).

**Zadanie:** Użyj otwarcia morfologicznego i usuwania małych obiektów (`remove_small_objects`), by pozostawić same główne struktury komórkowe.

In [ ]:
# WYKONAJ: Oczyszczenie maski (otwarcie + remove_small_objects)
# === TWÓJ KÓD TUTAJ ===
clean_mask = binary_mask # <- Podmień tę zmienną!
# ======================

# plt.title('Maska po operacjach morfologicznych')
# plt.imshow(clean_mask, cmap='gray')
# plt.show()

### --- TEORIA: Jak rozciąć stykające się komórki? ---
Dwie komórki stykają się ze sobą. Po zwykłym progowaniu Otsu tworzą jedną wielką, zlaną "wyspę". Komputer widzi to jako jeden obiekt. Aby je rozdzielić, użyjemy **Algorytmu Wododziałowego (Watershed)**.

Wyobraź sobie maskę binarną jako **trójwymiarową mapę terenu**:
1. **Transformata odległości (`ndi.distance_transform_edt`):** Funkcja ta oblicza odległość każdego białego piksela komórki od czarnego tła. Środki komórek (najdalej od krawędzi) zyskują najwyższe wartości, stając się "szczytami gór", a miejsca styku komórek (przewężenia) stają się "przełęczami".
2. **Szukanie markerów (`morphology.local_maxima` i `measure.label`):** Znajdujemy te szczyty i przypisujemy każdemu z nich unikalny numer (etykietę). Będą to nasze "źródła", z których zaczniemy wylewać wodę.
3. **Zalewanie i Wododział (`segmentation.watershed`):** Odwracamy mapę odległości (szczyty stają się głębokimi dolinami z unikalnymi źródłami na dnie). Woda zaczyna się podnosić. Gdy woda z lewej komórki (np. czerwona) spotka się z wodą z prawej komórki (np. niebieską) na przełęczy, algorytm buduje tam zaporę (wododział) – idealnie rozcinając obiekty!

### Krok 4: Algorytm Wododziałowy (Watershed)
**Zadanie:** Odkomentuj i przeanalizuj poniższy kod, korzystając z teorii powyżej.

In [ ]:
# 1. Transformata odległości (mapa odległości każdego punktu komórki do najbliższego tła)
# distance = ndi.distance_transform_edt(clean_mask)

# 2. Znalezienie szczytów (środków komórek) - z użyciem np. małego footprint (okręgu)
# local_maxi = morphology.local_maxima(distance)

# 3. Oznaczenie szczytów (ziaren) unikalnymi numerami (1, 2, 3...)
# markers = measure.label(local_maxi)

# 4. Właściwy algorytm watershed 
# Znak minus (-distance) odwraca mapę, aby szczyty stały się "dolinami" do zalewania.
# labels = segmentation.watershed(-distance, markers, mask=clean_mask)

# Wizualizacja przypisanych kolorów (zestawów)
# colored_cells = color.label2rgb(labels, bg_label=0)
# plt.title('Komórki rozdzielone algorytmem Watershed')
# plt.imshow(colored_cells)
# plt.show()

### --- TEORIA: Ekstrakcja Cech ---
Aby analizować parametry medyczne (np. średnicę komórek), używamy funkcji **`measure.regionprops(labels)`**. 
Funkcja ta iteruje po każdym ponumerowanym obiekcie w naszej masce `labels` (1, 2, 3...) i wylicza dla niego precyzyjne parametry, takie jak pole powierzchni (`area`), obwód, czy współrzędne środka.

### Krok 5: Raport i ekstrakcja cech medycznych
**Zadanie:** Na podstawie mapy znaczników (`labels`), użyj modułu `measure.regionprops()`, aby wygenerować raport. 

Oblicz i wypisz:
1. Całkowitą liczbę odseparowanych komórek.
2. Średnią powierzchnię (`area`) komórki w pikselach.
3. Maksymalną zmierzoną powierzchnię (największa komórka w rozmazie).

In [ ]:
# WYKONAJ: Generowanie raportu
# === TWÓJ KÓD TUTAJ ===

# ======================